# 04 — Self-supervised GNN Training
Trains a Graph Isomorphism Network (GIN) encoder using a self-supervised
contrastive learning framework (SimCLR / NT-Xent loss).

The model is trained on a large-scale dataset of approximately 5 million molecules
extracted from the [ZINC22 database](https://zinc22.docking.org/).
During training, two augmented views of each molecular graph are generated
via random edge dropout and node feature masking, and the encoder is optimized
to maximize agreement between views of the same molecule.

> **Note:** This step requires a GPU and significant training time (~5M molecules).
> Pre-trained model weights (`gnn_encoder.pt`) are available on Zenodo (see README).
> If you only want to reproduce the embeddings, skip to notebook `03_graph_embeddings.ipynb`.

## Configuration

In [ ]:
TRAINING_GRAPHS_PT = "data/training_graphs.pt"  # PyG graphs from ZINC22 (~5M molecules)
MODEL_OUTPUT_PT    = "data/gnn_encoder.pt"       # where to save trained weights

# Hyperparameters 
HIDDEN_DIM     = 128    # graph embedding dimensionality
NUM_LAYERS     = 3      # number of GIN message-passing layers
LR             = 1e-3   # learning rate
EPOCHS         = 10     # number of training epochs
BATCH_SIZE     = 128    # batch size
EDGE_DROPOUT   = 0.1    # probability of dropping each edge during augmentation
FEAT_MASK_PROB = 0.1    # probability of masking each node feature during augmentation
TEMPERATURE    = 0.1    # NT-Xent loss temperature

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Dependencies

In [ ]:
# pip install torch torch_geometric tqdm numpy

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from tqdm import tqdm
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool

## 2. Graph augmentation functions
Two stochastic augmentations are applied independently to each graph to generate
two views for the contrastive objective:
- **Edge dropout**: randomly removes a fraction of bonds
- **Node feature masking**: randomly zeros a fraction of atom feature vectors

In [ ]:
def dropout_edges(data: Data, drop_prob: float) -> Data:
    """Randomly drop edges with probability drop_prob."""
    if data.edge_index.numel() == 0 or drop_prob <= 0.0:
        return data
    mask = torch.rand(data.edge_index.size(1),
                      device=data.edge_index.device) > drop_prob
    new_data = data.clone()
    new_data.edge_index = data.edge_index[:, mask]
    return new_data


def mask_node_features(data: Data, mask_prob: float) -> Data:
    """Randomly zero node features with probability mask_prob."""
    if mask_prob <= 0.0:
        return data
    x    = data.x.clone()
    mask = torch.rand(x.size(0), device=x.device) < mask_prob
    x[mask] = 0.0
    new_data = data.clone()
    new_data.x = x
    return new_data


def graph_augment(data: Data) -> Data:
    """Apply edge dropout and node feature masking."""
    return mask_node_features(
        dropout_edges(data, EDGE_DROPOUT), FEAT_MASK_PROB
    )

## 3. GNN Encoder (GIN)
The encoder consists of `NUM_LAYERS` GIN convolution layers, each followed by
batch normalization and ReLU activation. A global additive pooling operation
aggregates node-level features into a fixed-dimensional graph embedding.
A two-layer projection head maps the embedding to the contrastive loss space.

In [ ]:
class GNNEncoder(nn.Module):
    """
    GIN-based graph encoder for self-supervised contrastive learning.
    input : x, edge_index, batch
    output: g — graph-level embedding (dim=HIDDEN_DIM)
            z — projection head output (for NT-Xent loss)
    """
    def __init__(self, in_dim: int,
                 hidden_dim: int = HIDDEN_DIM,
                 num_layers: int = NUM_LAYERS):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        dim = in_dim
        for _ in range(num_layers):
            self.convs.append(GINConv(nn.Sequential(
                nn.Linear(dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            dim = hidden_dim
        self.proj_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
        g = global_add_pool(x, batch)   # (N_graphs, hidden_dim)
        z = self.proj_head(g)
        return g, z


print('GNNEncoder defined.')

## 4. NT-Xent contrastive loss (SimCLR)
Given two sets of projections `z1` and `z2` from augmented views of the same
batch of graphs, the loss encourages high similarity between matching pairs
and low similarity between non-matching pairs.

In [ ]:
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor,
                 temperature: float = TEMPERATURE) -> torch.Tensor:
    """
    Normalized temperature-scaled cross-entropy loss.
    z1, z2: (B, D) — projections from two augmented views.
    """
    B    = z1.size(0)
    z1   = F.normalize(z1, dim=1)
    z2   = F.normalize(z2, dim=1)
    reps = torch.cat([z1, z2], dim=0)                    # (2B, D)
    sim  = reps @ reps.t()                               # (2B, 2B)
    # Mask self-similarities
    sim.masked_fill_(torch.eye(2*B, device=sim.device, dtype=torch.bool), -9e15)
    # Positive pairs: (i, i+B) and (i+B, i)
    positives = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)
    loss = -torch.log(
        torch.exp(positives / temperature) /
        torch.exp(sim / temperature).sum(dim=1)
    ).mean()
    return loss

## 5. Dataset wrapper

In [ ]:
class GraphDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

## 6. Training loop

In [ ]:
def train_contrastive(dataset: GraphDataset) -> GNNEncoder:
    loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    in_dim    = dataset[0].x.size(1)
    model     = GNNEncoder(in_dim=in_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print('=========== TRAINING SELF-SUPERVISED ===========')
    print(f'Device       : {DEVICE}')
    print(f'Graphs       : {len(dataset)}')
    print(f'Batch size   : {BATCH_SIZE}')
    print(f'Epochs       : {EPOCHS}')
    print(f'Hidden dim   : {HIDDEN_DIM}')
    print(f'Layers       : {NUM_LAYERS}')
    print(f'Temperature  : {TEMPERATURE}')
    print('================================================\n')

    model.train()
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        total_loss, n_batches = 0.0, 0

        for batch in tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
            batch = batch.to(DEVICE)
            b1    = graph_augment(batch)
            b2    = graph_augment(batch)
            _, z1 = model(b1.x, b1.edge_index, b1.batch)
            _, z2 = model(b2.x, b2.edge_index, b2.batch)
            loss  = nt_xent_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(1, n_batches)
        print(f'Epoch {epoch:3d}/{EPOCHS} | loss={avg_loss:.4f} | {time.time()-t0:.1f}s')

    print('\n============== TRAINING COMPLETE ================')
    return model

## 7. Run training and save model

In [ ]:
graphs  = torch.load(TRAINING_GRAPHS_PT, weights_only=False)
dataset = GraphDataset(graphs)

model = train_contrastive(dataset)

torch.save(model.state_dict(), MODEL_OUTPUT_PT)
print(f'Model weights saved: {MODEL_OUTPUT_PT}')

## 8. Quick sanity check
Verify that the saved weights load correctly and produce embeddings of the expected shape.

In [ ]:
in_dim      = graphs[0].x.size(1)
model_check = GNNEncoder(in_dim=in_dim).to(DEVICE)
model_check.load_state_dict(torch.load(MODEL_OUTPUT_PT,
                                        weights_only=False,
                                        map_location=DEVICE))
model_check.eval()

sample = graphs[0].to(DEVICE)
with torch.no_grad():
    g, z = model_check(sample.x,
                       sample.edge_index,
                       torch.zeros(sample.x.size(0), dtype=torch.long, device=DEVICE))

print(f'Graph embedding shape    : {g.shape}   (expected: [1, {HIDDEN_DIM}])')
print(f'Projection output shape  : {z.shape}   (expected: [1, {HIDDEN_DIM}])')
print('Sanity check passed.')